# Week 2 - Overview of Lyapunov Stability Theory

**<ins>Motivation</ins>**

In nonlinear spacecraft dynamics, the equations of motion are rarely solvable in closed form.  
Linearization can provide local insight, but it does not guarantee behavior for large-angle maneuvers or tumbling motion.

Lyapunov’s Direct Method provides a systematic way to study stability **without solving the differential equations explicitly**.  
Instead of computing trajectories, we construct a scalar function that behaves like an energy measure and analyze how it evolves over time.

If this function decreases appropriately, we can prove stability and, in stronger cases, convergence.

This method forms the foundation for modern nonlinear spacecraft attitude control design.


**<ins>Core Ideas</ins>**

1) **Function Definiteness**

The starting point is understanding positive definite, negative definite, and semi-definite functions.

A positive definite function behaves like stored energy:
- It is zero at the equilibrium.
- It is strictly positive elsewhere.

If the time derivative of this function is negative definite, the system dissipates “energy,” which implies convergence.

These definiteness properties are the mathematical backbone of Lyapunov analysis.


2) **Lyapunov Candidate Functions**

For mechanical and spacecraft systems, convenient prototype functions are available:

- **Rate-based functions**  
  Built from angular velocity error (kinetic energy–like).

- **State-based functions**  
  Built from attitude error (potential energy–like).

These prototype structures will later lead directly to nonlinear attitude control laws.


**<ins>Key Takeaways</ins>**

By the end of this week, you should be able to:

1. Apply Lyapunov’s direct method to argue stability and convergence for nonlinear dynamical systems.
2. Clearly distinguish between stability, asymptotic stability, exponential stability, and global versus local stability.
3. Understand how Lyapunov theory connects directly to spacecraft attitude control design.

This week builds the theoretical stability framework that the remainder of nonlinear spacecraft control relies on.

---

In [1]:
import sympy as sp
sp.init_printing(use_latex='mathjax')
from IPython.display import display, Math

import numpy as np

# 2.1 - Stability Certificates and Control Law Design

## 2.1.1 - Why a Direct Method is needed

Start with a nonlinear system

$$
\dot{x} = f(x)
$$

We want to know: if the system is slightly disturbed from equilibrium, what happens?

The obvious idea is: solve the differential equation and look at $x(t)$.  
But for nonlinear systems, this is basically unrealistic. Closed-form solutions rarely exist. Numerical simulations only test specific initial conditions. Stability, however, is a statement about *all nearby initial conditions*. Simulation cannot give a general guarantee.

The next idea is linearization. Around an equilibrium,

$$
\dot{\delta x} = A \delta x
$$

and then check eigenvalues of $A$. If the real parts are negative, we get local asymptotic stability.

This works — but only locally. It studies an approximation of the system. Away from equilibrium, nonlinear effects can change everything. This approach is called the **indirect method** because we infer stability from a linear approximation rather than from the original nonlinear system itself.

Lyapunov’s idea was different.

Instead of trying to compute where the state goes, ask:

**Is there some scalar quantity that the system can never increase?**

This is the key shift.

We introduce a scalar function $V(x)$. Think of it as an artificial energy or a measure of how far we are from equilibrium.

We do not claim this is the true physical energy.  
We only require that it behaves like one.

If we can find a function $V(x)$ such that:

- $V(x) > 0$ whenever $x \neq 0$  
- and $\dot{V}(x) \le 0$ along system trajectories  

then something powerful happens.

The system cannot gain “energy.”  
The motion is trapped inside level sets of $V(x)$.  
The state cannot escape without violating the inequality.

Notice what we did **not** do:

- We did not solve the differential equation.
- We did not linearize.
- We did not approximate.

We constructed a **certificate**.

If such a function exists, stability follows automatically from the inequalities it satisfies.

That is why this is called the **direct method**.

It works directly with the nonlinear system and proves stability through a scalar inequality, not through explicit solutions.

The deep idea is simple:

If a scalar function is bounded below and monotonically decreasing, it must converge.

Lyapunov’s method applies this basic fact to dynamical systems.

We are not predicting motion.  
We are restricting motion.

## 2.1.2 - First Condition of Lyapunov’s Direct Method — Positive Definiteness

Lyapunov’s Direct Method rests on two structural conditions.  
The first is geometric. The second will be dynamical.

Before asking how a scalar function evolves over time, we must ensure that it actually behaves like a meaningful measure of deviation from equilibrium.

Assume the equilibrium has been shifted to the origin.

For a scalar function $V(x)$ to qualify as a Lyapunov candidate, it must satisfy

$$
V(0) = 0,
$$

and

$$
V(x) > 0 \quad \forall x \neq 0 \text{ in some neighborhood of } 0.
$$

This property is called **positive definiteness**.

Geometrically, this means the origin is a strict local minimum. If we move away from equilibrium in any direction, the value of $V(x)$ increases. The level sets of $V(x)$ form closed contours around the origin. These closed contours are important because they will later act as regions that trajectories cannot escape from.

If instead

$$
V(0) = 0, \quad V(x) \ge 0,
$$

then $V(x)$ is **positive semi-definite**. The function never becomes negative, but it may be flat in certain directions. Along those directions, $V(x)$ does not strictly measure deviation from equilibrium. This distinction becomes important when discussing convergence.

If $V(x)$ takes both positive and negative values arbitrarily close to the origin, it is **indefinite**. Such a function cannot serve as a stability certificate because it does not consistently represent deviation from equilibrium.

Definiteness may hold only inside a neighborhood of the origin (local definiteness), or for all $x \in \mathbb{R}^n$ (global definiteness). This distinction will later correspond directly to local versus global stability conclusions.

In many engineering applications, Lyapunov candidates are chosen as quadratic forms:

$$
V(x) = x^T P x,
$$

where $P$ is symmetric.

The definiteness of $V(x)$ is determined entirely by the matrix $P$.

- All eigenvalues of $P$ strictly positive $\Rightarrow$ $V(x)$ is positive definite.
- All eigenvalues non-negative $\Rightarrow$ positive semi-definite.
- Mixed signs $\Rightarrow$ indefinite.

To see why eigenvalues matter, consider an eigenvector $v$ of $P$ with eigenvalue $\lambda$. Along that direction,

$$
V(\alpha v) = \alpha^2 \lambda.
$$

If $\lambda > 0$, the function increases as we move away from the origin in that direction. If $\lambda < 0$, it decreases. If $\lambda = 0$, it remains flat.

Thus, positive definiteness guarantees that the function curves upward in every direction — exactly the “bowl” behavior required for a stability certificate.

At this stage, any function satisfying the positive definiteness condition is called a **Lyapunov candidate function**.

The word “candidate” is important. Positive definiteness alone does not prove stability. It only satisfies the first structural requirement of the Direct Method.

A function may look like a perfect energy well, but if its value increases along system trajectories, it cannot certify stability.

The decisive step is to examine how $V(x)$ evolves over time. This requires computing its time derivative along trajectories,

$$
\dot{V}(x) = \nabla V(x)^T f(x),
$$

and analyzing its sign.

This forms the second structural condition of Lyapunov’s Direct Method covered in section 2.1.3.

In [2]:
def check_definiteness(obj, variables=None, equilibrium=None, sample_points=100):
    """
    Check definiteness of a matrix (NumPy) or a function (SymPy).
    
    Parameters
    ----------
    obj : numpy.ndarray or sympy.Expr
        The matrix or scalar function to check.
        
    variables : list of sympy.Symbol, optional
        Variables used in the function (only needed if obj is SymPy expression).
        
    equilibrium : list/tuple of floats, optional
        Equilibrium point around which to test (default = all zeros).
        
    sample_points : int
        Number of random samples near equilibrium for numeric testing (function case).
    
    Returns
    -------
    str : Definiteness classification
    """
    
    # Case 1: Matrix
    if isinstance(obj, np.ndarray):
        eigvals = np.linalg.eigvals(obj)
        
        print("Matrix:\n", obj)
        print(f"Eigenvalues: {eigvals}")
        
        if np.all(eigvals > 0):
            return "Positive definite"
        elif np.all(eigvals >= 0):
            return "Positive semi-definite"
        elif np.all(eigvals < 0):
            return "Negative definite"
        elif np.all(eigvals <= 0):
            return "Negative semi-definite"
        else:
            return "Indefinite"
    
    # Case 2: Symbolic function (expressed using SymPy)
    elif isinstance(obj, sp.Expr):
        display("Function V(x):", obj)
        
        if variables is None:
            raise ValueError("Must provide variables for SymPy function")
        if equilibrium is None:
            equilibrium = [0]*len(variables)
        
        # Check equilibrium point
        V_eq = obj.subs(dict(zip(variables, equilibrium)))
        if V_eq != 0:
            return f"Not a valid Lyapunov candidate (V(eq) = {V_eq})"
        
        # Sample near equilibrium
        vals = []
        for _ in range(sample_points):
            rand_point = np.random.uniform(-1, 1, len(variables)) * 0.5
            V_sub = sp.N(obj.subs(dict(zip(variables, rand_point))))
            V_val = float(V_sub)
            vals.append(V_val)
        
        if all(v > 0 for v in vals):
            return "Positive definite (numerical check)"
        elif all(v >= 0 for v in vals):
            return "Positive semi-definite (numerical check)"
        elif all(v < 0 for v in vals):
            return "Negative definite (numerical check)"
        elif all(v <= 0 for v in vals):
            return "Negative semi-definite (numerical check)"
        else:
            return "Indefinite (numerical check)"
    
    else:
        raise TypeError("Input must be a NumPy matrix or SymPy expression")

# Example Usage:

# Case 1: Inputting a matrix
print("Case 1:")
K = np.array([[-1, 0], 
              [0, 0]])
print(check_definiteness(K))

print()

#Case 2: Inputting a function V(x)
print("Case 2:")
x, xdot = sp.symbols('x xdot')
V = 0.5*x**2 #+ 0.5*xdot**2
print(check_definiteness(V, variables=[x, xdot]))

Case 1:
Matrix:
 [[-1  0]
 [ 0  0]]
Eigenvalues: [-1.  0.]
Negative semi-definite

Case 2:


'Function V(x):'

     2
0.5⋅x 

Positive definite (numerical check)


In [3]:
# Concept Check 1 - Qn1
x1, x2 = sp.symbols('x1 x2')
V = 0.5*(x1**2 + x2**2)
print(check_definiteness(V, variables=[x1, x2]))

'Function V(x):'

      2         2
0.5⋅x₁  + 0.5⋅x₂ 

Positive definite (numerical check)


In [4]:
# Concept Check 1 - Qn2
x1, x2 = sp.symbols('x1 x2')
V = 0.5*(x1**2 - x2**2)
print(check_definiteness(V, variables=[x1, x2]))

'Function V(x):'

      2         2
0.5⋅x₁  - 0.5⋅x₂ 

Indefinite (numerical check)


In [5]:
# Concept Check 1 - Qn3
x1, x2 = sp.symbols('x1 x2')
V = sp.log(1 + x1**2 + x2**2)
print(check_definiteness(V, variables=[x1, x2]))

'Function V(x):'

   ⎛  2     2    ⎞
log⎝x₁  + x₂  + 1⎠

Positive definite (numerical check)


In [6]:
# Concept Check 1 - Qn4
x1, x2 = sp.symbols('x1 x2')
V = 0.5*(x1**2 + 4*x2**2)
print(check_definiteness(V, variables=[x1, x2]))

'Function V(x):'

      2         2
0.5⋅x₁  + 2.0⋅x₂ 

Positive definite (numerical check)


In [7]:
# Concept Check 1 - Qn5
x1, x2 = sp.symbols('x1 x2')
V = 0.5*(x1**2 + 4*x2**2) * sp.exp(-(x1**2 + 4*x2**2))
print(check_definiteness(V, variables=[x1, x2]))

'Function V(x):'

                         2       2
⎛      2         2⎞  - x₁  - 4⋅x₂ 
⎝0.5⋅x₁  + 2.0⋅x₂ ⎠⋅ℯ             

Positive definite (numerical check)


In [8]:
# Concept Check 1 - Qn6
K = np.array([[1.53947, -0.0422688, -0.190629], 
              [-0.0422688, 1.4759, 0.459006],
              [-0.190629, 0.459006, 1.48463]])
print(check_definiteness(K))

Matrix:
 [[ 1.53947   -0.0422688 -0.190629 ]
 [-0.0422688  1.4759     0.459006 ]
 [-0.190629   0.459006   1.48463  ]]
Eigenvalues: [1.99999822 1.50000363 0.99999815]
Positive definite


In [9]:
# Concept Check 1 - Qn7
K = np.array([[-0.984331, -1.10006, -0.478579], 
              [-1.10006, 1.03255, 0.338318],
              [-0.478579, 0.338318, 1.45178]])
print(check_definiteness(K))

Matrix:
 [[-0.984331 -1.10006  -0.478579]
 [-1.10006   1.03255   0.338318]
 [-0.478579  0.338318  1.45178 ]]
Eigenvalues: [-1.49999901  1.99999849  0.99999952]
Indefinite


In [10]:
# Concept Check 1 - Qn8
K = np.array([[-2.0353, 0.296916, -0.365128], 
              [0.296196, -1.10369, -0.074481],
              [-0.365128, -0.074481, -2.86101]])
print(check_definiteness(K))

Matrix:
 [[-2.0353    0.296916 -0.365128]
 [ 0.296196 -1.10369  -0.074481]
 [-0.365128 -0.074481 -2.86101 ]]
Eigenvalues: [-2.99999716 -1.9997948  -1.00020804]
Negative definite


## 2.1.3 - Second Condition of Lyapunov’s Direct Method — Time Derivative and Stability

The first structural condition ensured that $V(x)$ behaves like an energy well around the equilibrium.  
It guarantees geometry.

We now impose the second condition: how this energy changes as the system evolves.

Consider the nonlinear autonomous system

$$
\dot{x} = f(x),
$$

and let $V(x)$ be a continuously differentiable Lyapunov candidate.

Since the state evolves as $x = x(t)$, the scalar function $V(x)$ becomes a time function

$$
V(t) = V(x(t)).
$$

To understand how the “energy” changes, we compute its time derivative along trajectories.

By the chain rule,

$$
\dot V(x) = \frac{d}{dt} V(x(t)) = \nabla V(x)^T \dot x.
$$

Substituting $\dot x = f(x)$ gives

$$
\dot V(x) = \nabla V(x)^T f(x).
$$

This expression is the key object in Lyapunov analysis.

It tells us whether the system moves uphill or downhill on the Lyapunov surface.

- If $\dot V(x) > 0$, the system climbs the surface.
- If $\dot V(x) = 0$, the motion is tangent to a level set.
- If $\dot V(x) < 0$, the system descends.

This leads directly to the central implication of the Direct Method.

If there exists a neighborhood of the origin such that

- $V(x)$ is positive definite, and  
- $\dot V(x) \le 0$,

then the equilibrium is **Lyapunov stable**.

The reasoning is purely analytic.

Since $V(x(t))$ is non-increasing and bounded below by zero, it can never grow unbounded.  
Therefore, trajectories that start close to the origin remain confined within a level set of $V$.

If, in addition,

$$
\dot V(x) < 0 \quad \forall x \neq 0,
$$

then the equilibrium is **asymptotically stable**.

Here the energy strictly decreases whenever the system is away from equilibrium.  
Because $V(x(t))$ is decreasing and bounded below, it must converge.  
The only state where decrease can stop is the equilibrium itself.


**When $\dot V$ is only semi-definite**

In many nonlinear systems we obtain

$$
\dot V(x) \le 0,
$$

but

$$
\dot V(x) = 0
$$

on a nontrivial set

$$
\Omega = \{ x \mid \dot V(x) = 0 \}.
$$

In this situation, stability is guaranteed, but convergence is not obvious.

Geometrically, this corresponds to a flat region on the Lyapunov surface.  
The system rolls downhill and may reach a plateau where the first derivative vanishes.  
Once there, the first derivative no longer forces further descent.

To determine whether the system can remain in this flat region, we examine the behavior of

$$
V(t) = V(x(t))
$$

more carefully.

Even if

$$
\dot V(t) = 0,
$$

this does not mean the function is constant in time.  
It only means the slope is zero at that instant.

We can therefore examine higher time derivatives:

$$
\ddot V(t), \quad \frac{d^3}{dt^3} V(t), \quad \dots
$$

These derivatives describe how the slope itself changes.

Using a Taylor expansion of $V(t)$ about some time $t_0$,

$$
V(t) = V(t_0)
+ \frac{1}{k!} V^{(k)}(t_0) (t - t_0)^k
+ \dots
$$

where $V^{(k)}$ denotes the $k$-th time derivative.

If the first derivative is zero, and the second derivative is also zero, and so on up to order $k-1$, but the $k$-th derivative is the first nonzero term, then that term determines the local behavior.

If that first nonzero derivative is:

- of **even order**, it describes curvature (like a parabola).  
- of **odd order**, it introduces directional slope.

For forward-time motion, we need directional slope.

This is the idea behind the Mukherjee–Chen refinement.

Formally:

Suppose $V(x)$ is positive definite and $\dot V(x) \le 0$.  
Let $\Omega = \{ x \mid \dot V(x) = 0 \}$.

If, along trajectories confined to $\Omega$,

- $\frac{d^i}{dt^i} V(x(t)) = 0$ for $i = 1, 2, \dots, k-1$, and  
- the first nonzero derivative $\frac{d^k}{dt^k} V(x(t))$ is negative definite with $k$ odd,

then trajectories cannot remain in $\Omega$ unless they are at equilibrium.

The meaning of this condition is simple:

The energy function may appear flat at first order, but higher-order derivatives reveal hidden descent.  
Because the first nonzero derivative is of odd order and negative, the Taylor expansion forces $V(t)$ to decrease for small positive time.

Thus the system cannot persist on the plateau.  
It must eventually descend toward equilibrium.

In this way, higher-order analysis restores asymptotic stability even when $\dot V$ is only semi-definite.


The second structural condition therefore connects geometry to motion:

- Positive definiteness shapes the energy well.
- The sign of $\dot V$ determines whether motion is confined or convergent.
- Higher-order derivatives detect hidden descent when first-order analysis is inconclusive.

Together with the first structural condition, this completes the logical core of Lyapunov’s Direct Method.

In [12]:
# Concept Check 2 - Qn2

# Define symbols
t, k = sp.symbols('t k', positive=True, real=True)
x = sp.Function('x')(t)
xdot = sp.diff(x, t)
xddot = sp.diff(x, t, 2)

# Candidate Lyapunov function
V = sp.Rational(1, 2)*xdot**2 + k*sp.Rational(1, 4)*x**4
display(Math(r"V(x,\dot{x}) = " + sp.latex(V)))

# Compute Vdot = dV/dx * xdot + dV/dxdot * xddot
dV_dx = sp.diff(V, x)
dV_dxdot = sp.diff(V, xdot)
Vdot = sp.Mul(dV_dx, xdot) + sp.Mul(dV_dxdot, xddot)

# Substitute system dynamics: xddot = -k*x^3
Vdot_sub = sp.simplify(Vdot.subs(xddot, -k*x**3))
display(Math(r"\dot{V}(x,\dot{x}) = " + sp.latex(Vdot_sub)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [ ]:
# Concept Check 3 - Qn1
x, xdot, alpha = sp.symbols('x xdot alpha', real=True, positive=True)
Vdot = -alpha * xdot**2
display(Math(r"\dot{V}(x,\dot{x}) = " + sp.latex(Vdot)))

Vdot_sub = Vdot.subs(alpha, 1)
print(check_definiteness(Vdot_sub, variables=[x, xdot], equilibrium=[0, 0]))

In [ ]:
# Concept Check 3 - Qn2
x, xdot = sp.symbols('x xdot', real=True, positive=True)
Vdot = -alpha * xdot**2
display(Math(r"\dot{V}(x,\dot{x}) = " + sp.latex(Vdot)))

Vdot_sub = Vdot.subs(alpha, 1)
print(check_definiteness(Vdot_sub, variables=[x, xdot], equilibrium=[0, 0]))

## 2.1.4 - Local vs Global Lyapunov Stability

Lyapunov stability does not depend only on the form of the Lyapunov function.  
It also depends on **where in the state space the Lyapunov conditions hold**.

For nonlinear systems, the conditions

$$
V(x) > 0, \quad \dot V(x) \le 0
$$

are often satisfied only within a limited region around the equilibrium.

This leads to an important distinction between **local** and **global** stability.

**<ins>Local Lyapunov Stability</ins>**

An equilibrium point $x_r$ is **locally Lyapunov stable** if there exists a neighborhood

$$
B_\delta(x_r)
$$

such that within this neighborhood

- $V(x)$ is positive definite, and  
- $\dot V(x) \le 0$.

Under these conditions, trajectories that start sufficiently close to the equilibrium remain close for all future time.

In geometric terms, the Lyapunov function defines level sets

$$
\Omega_c = \{x \mid V(x) \le c\}.
$$

Because $\dot V(x) \le 0$, the system cannot move to higher values of $V$.  
Once a trajectory enters a level set of $V$, it cannot leave it.

This creates an **invariant region** around the equilibrium.

However, the guarantee only holds inside the neighborhood where the Lyapunov conditions are satisfied.  
Outside that region:

- $V(x)$ may lose definiteness  
- $\dot V(x)$ may change sign  
- nonlinear dynamics may dominate

As a result, trajectories far from equilibrium may behave very differently.

This is the typical situation for nonlinear systems: stability can often be certified **locally**, but not globally.


**<ins>Global Lyapunov Stability</ins>**

An equilibrium point $x_r$ is **globally Lyapunov stable** if the Lyapunov conditions hold for all states

$$
x \in \mathbb{R}^n.
$$

That is,

- $V(x)$ is positive definite everywhere, and  
- $\dot V(x) \le 0$ everywhere.

If, in addition,

$$
\dot V(x) < 0 \quad \forall x \ne x_r,
$$

then the equilibrium is **globally asymptotically stable**.

This means that **all trajectories in the state space remain bounded and converge to the equilibrium**.


**<ins>Radial Unboundedness</ins>**

For global asymptotic stability, one additional property is usually required.

The Lyapunov function must be **radially unbounded**, meaning

$$
\|x\| \to \infty \quad \Rightarrow \quad V(x) \to \infty.
$$

This condition ensures that the energy function grows without bound as we move farther away from the equilibrium.

The consequence is that the level sets

$$
\Omega_c = \{x \mid V(x) \le c\}
$$

remain bounded.

Without radial unboundedness, it is possible for $V(x)$ to remain bounded even while $\|x\|$ becomes arbitrarily large.  
In such cases, trajectories could escape to infinity while the Lyapunov function remains constant or decreases.

Radial unboundedness prevents this possibility by ensuring that the Lyapunov function behaves like a true global energy measure.


**<ins>Why This Distinction Matters</ins>**

For **linear time-invariant systems**, this distinction rarely appears.  
Stable linear systems admit quadratic Lyapunov functions that are globally valid, so stability results extend across the entire state space.

Nonlinear systems behave differently.

Even when stability can be proven near the equilibrium, nonlinear effects may dominate far from it.  
Trajectories that decay locally may diverge, escape to infinity, or converge to other attractors elsewhere in the state space.

The phase portraits in the lecture slides illustrate this clearly:  
behavior near equilibrium does not necessarily reflect behavior globally.


**<ins>Key Takeaway</ins>**

Lyapunov stability depends on both

- the existence of a suitable Lyapunov function, and  
- the **region of the state space where its conditions hold**.

Local Lyapunov stability guarantees bounded behavior near the equilibrium.

Global Lyapunov stability guarantees stability for all initial conditions.

For nonlinear systems, stability results are often local by necessity.  
Achieving global guarantees requires stronger conditions, such as global definiteness and radial unboundedness of the Lyapunov function.

## 2.1.5 - Linear System through the Lyapunov Lens

Lyapunov’s Direct Method should not be viewed as an alternative to classical linear stability theory.  
Instead, it provides an equivalent formulation that naturally generalizes to nonlinear systems.

Consider the linear time-invariant system

$$
\dot{x} = A x.
$$

Classical linear systems theory states that the equilibrium $x = 0$ is asymptotically stable if and only if all eigenvalues of $A$ have strictly negative real parts. Such a matrix is called **Hurwitz**.

Lyapunov theory reproduces this result using an energy-based argument.


**<ins>Quadratic Lyapunov Function</ins>**

For linear systems, a natural Lyapunov candidate is a quadratic function

$$
V(x) = x^T P x
$$

where $P = P^T$ is a symmetric matrix.

This function behaves like a generalized energy of the state vector.

For $V(x)$ to measure distance from the equilibrium, it must be positive definite.  
This occurs when

$$
P > 0
$$

meaning

$$
x^T P x > 0 \quad \forall x \neq 0.
$$


**<ins>Time Derivative Along System Trajectories</ins>**

To determine whether this energy increases or decreases, we compute the time derivative of $V(x)$ along system trajectories.

Since

$$
V(x) = x^T P x
$$

and both $x$ terms depend on time, the derivative must be applied to both factors.

Using the product rule,

$$
\dot V = \frac{d}{dt}(x^T P x) = \dot x^T P x + x^T P \dot x.
$$

Substituting the system dynamics

$$
\dot x = A x
$$

gives

$$
\dot V = (A x)^T P x + x^T P (A x).
$$

Now expand the transpose term.

Recall

$$
(Ax)^T = x^T A^T.
$$

Therefore

$$
\dot V = x^T A^T P x + x^T P A x.
$$

Factor out the state vector:

$$
\dot V = x^T (A^T P + P A) x.
$$

This expression shows that the sign of $\dot V$ depends entirely on the matrix

$$
A^T P + P A.
$$


**<ins>The Lyapunov Equation</ins>**

For asymptotic stability we require

$$
\dot V(x) < 0 \quad \forall x \neq 0.
$$

From the expression above, this occurs when

$$
A^T P + P A
$$

is negative definite.

It is convenient to express this condition by introducing a positive definite matrix $Q$ and writing

$$
A^T P + P A = -Q.
$$

This matrix equation is known as the **continuous Lyapunov equation**.

Substituting this into the derivative gives

$$
\dot V = x^T(-Q)x = -x^T Q x.
$$

Since $Q > 0$, we obtain

$$
\dot V < 0 \quad \forall x \neq 0.
$$

Thus the quadratic function $V(x) = x^T P x$ satisfies the Lyapunov conditions and proves asymptotic stability.


**<ins>Equivalence with Eigenvalue-Based Stability</ins>**

A fundamental theorem in control theory states:

The matrix $A$ is Hurwitz **if and only if** for every symmetric positive definite matrix $Q$ there exists a unique symmetric positive definite matrix $P$ satisfying

$$
A^T P + P A = -Q.
$$

This result shows that Lyapunov’s method is completely consistent with classical linear stability theory.

If $A$ is stable, a quadratic Lyapunov function always exists.  
If such a Lyapunov function exists, $A$ must be stable.


**<ins>Global Stability of Linear Systems</ins>**

For linear systems this stability result is automatically global.

Because the dynamics

$$
\dot x = A x
$$

are homogeneous in the state, the same stability behavior holds regardless of the magnitude of $x$.

A quadratic Lyapunov function therefore certifies stability over the entire state space

$$
x \in \mathbb{R}^n.
$$

This contrasts with nonlinear systems, where Lyapunov functions often certify stability only within a limited region around the equilibrium.


**<ins>Key Takeaway</ins>**

For linear time-invariant systems,

$$
A \text{ is Hurwitz}
\quad \Longleftrightarrow \quad
\exists \, P>0 \text{ such that } V(x)=x^T P x
$$

is a valid Lyapunov function.

Lyapunov’s Direct Method therefore does not replace classical eigenvalue-based stability analysis.  
It provides a deeper viewpoint that naturally extends to nonlinear systems where eigenvalue methods no longer apply.

## 2.1.6 - Section 2.1 Summary

Lyapunov’s Direct Method provides a way to determine stability **without explicitly solving the system dynamics**.

Given a nonlinear dynamical system

$$
\dot{x} = f(x),
$$

the classical approach would attempt to compute the trajectory $x(t)$ and inspect its long–term behavior.  
For nonlinear systems this is rarely practical, since closed-form solutions seldom exist.

Lyapunov’s idea is to instead study the evolution of a scalar function

$$
V(x)
$$

that behaves like an energy measure of the system state.

If this function is constructed carefully, the stability of the equilibrium can be inferred purely from the properties of $V(x)$ and its time derivative.

---

**<ins>The Two Structural Conditions of Lyapunov’s Direct Method</ins>**

The method relies on two fundamental conditions.

First, the function must behave like a meaningful measure of deviation from equilibrium.

This requires **positive definiteness**

$$
V(x) > 0 \quad \forall x \neq 0, \qquad V(0)=0.
$$

Geometrically, this places the equilibrium at the bottom of an energy well.

Second, we examine how this energy evolves along system trajectories.

Using the chain rule,

$$
\dot V(x) = \frac{d}{dt}V(x(t)) = \nabla V(x)^T f(x).
$$

The sign of $\dot V(x)$ determines the system behavior:

- If $\dot V(x) \le 0$, the system cannot gain energy and the equilibrium is **Lyapunov stable**.
- If $\dot V(x) < 0$ for all $x \neq 0$, the energy strictly decreases and the equilibrium is **asymptotically stable**.

Thus stability is inferred from **energy inequalities rather than explicit trajectories**.

---

**<ins>Local and Global Stability</ins>**

The strength of a Lyapunov result depends on the region of the state space where these conditions hold.

If the Lyapunov conditions hold only within a neighborhood of the equilibrium, the result guarantees **local stability**.

If they hold for all $x \in \mathbb{R}^n$, and the Lyapunov function is **radially unbounded**

$$
\|x\| \to \infty \quad \Rightarrow \quad V(x) \to \infty,
$$

then the equilibrium is **globally stable**.

Radial unboundedness ensures that the energy function grows without bound as the state moves farther away, preventing trajectories from escaping to infinity.

---

**<ins>Connection with Linear Stability Theory</ins>**

For linear time-invariant systems

$$
\dot{x} = A x,
$$

Lyapunov theory reproduces the classical eigenvalue stability condition.

A quadratic Lyapunov function

$$
V(x) = x^T P x, \qquad P = P^T > 0
$$

yields

$$
\dot V(x) = x^T (A^T P + P A) x.
$$

If there exists a matrix $P>0$ such that

$$
A^T P + P A = -Q, \qquad Q>0,
$$

then

$$
\dot V(x) = -x^T Q x < 0,
$$

which proves asymptotic stability.

A fundamental result states that such a matrix $P$ exists **if and only if** the matrix $A$ is Hurwitz, meaning all eigenvalues of $A$ have negative real parts.

Thus Lyapunov stability theory is fully consistent with classical linear stability analysis.

---

**<ins>Key Insight</ins>**

Lyapunov’s Direct Method replaces trajectory analysis with energy analysis.

Instead of solving the differential equations, stability can be certified by constructing a scalar function whose value decreases along system trajectories.

This perspective extends naturally to nonlinear systems, where eigenvalue-based methods alone are no longer sufficient.

# 2.2 - Designing Lyapunov Functions for Control

Lyapunov’s direct method tells us:

If we can find a scalar function $V(x)$ that behaves like energy and never increases, then the system is stable.

That sounds simple.

The hard part is this:

How do we actually *find* such a function?

For a general nonlinear system

$$
\dot{x} = f(x),
$$

there is no universal formula that spits out a Lyapunov function.  
In theory, any scalar function could work.  
In practice, randomly guessing functions almost never works.

This is where **prototype Lyapunov functions** come in.

A prototype Lyapunov function is not a finished solution.  
It is a **structured template** — a starting shape that is known to work well for a certain class of systems.

Instead of asking:

> “What function should I invent?”

we ask:

> “What structure does physics already give me?”

For mechanical and spacecraft systems, two quantities naturally appear:

- Energy stored in motion (rates),
- Energy stored in configuration (position or attitude).

These energy expressions already have the properties we want:

- They are non-negative.
- They are zero at meaningful equilibria.
- Their time derivatives tell us whether energy is being dissipated or conserved.

So rather than inventing Lyapunov functions from scratch, we begin with familiar energy-like expressions and modify them to match our control objective.

Prototype functions are introduced gradually:

- First, functions that depend only on rates (useful for removing motion).
- Then, functions that depend on configuration.
- Finally, combinations of both, which are needed for full stabilization.

Each prototype answers a specific question:

- Are we trying to remove motion?
- Are we trying to regulate position?
- Are we trying to do both?

The term “prototype” simply means:

We reuse structural patterns that repeatedly succeed, instead of reinventing Lyapunov functions from nothing.

Understanding these patterns is what turns Lyapunov analysis from an abstract theorem into a practical control design tool.

## 2.2.1 - Rate-Based Lyapunov Functions

The first practical Lyapunov prototype answers a focused question:

Can we eliminate motion in a provably stable way?

Many control tasks begin with this objective:
- detumbling a spacecraft,
- damping vibrations,
- removing unwanted oscillations.

In all of these, the immediate goal is

$$
\dot q \rightarrow 0,
$$

without yet specifying where the configuration $q$ should settle.

For mechanical systems, this leads naturally to velocity-based Lyapunov functions.

### 2.2.1.1 - Regulation Problem

Consider a natural mechanical system with generalized coordinates $q \in \mathbb{R}^n$ and state $(q,\dot q)$.  
Its kinetic energy can always be written as

$$
T = \frac{1}{2}\dot q^T M(q)\dot q,
\qquad M(q)=M^T(q)>0.
$$

This suggests a natural Lyapunov candidate:

$$
V(\dot q) = \frac{1}{2}\dot q^T M(q)\dot q.
$$

This function is:
- positive definite in $\dot q$,
- zero if and only if $\dot q = 0$,
- unbounded as $\|\dot q\| \to \infty$.

It measures “how much motion” the system has.

Taking the time derivative along system trajectories gives

$$
\dot V = \dot q^T M(q)\ddot q + \frac{1}{2}\dot q^T \dot M(q)\dot q.
$$

A key structural identity of mechanical systems causes all internal velocity-coupling terms to cancel, leaving

$$
\dot V = \dot q^T Q,
$$

where $Q$ is the generalized force vector.

This is the work–energy relationship:  
the rate of change of kinetic energy equals applied power.

If we choose dissipative feedback

$$
Q = -P \dot q, \qquad P=P^T>0,
$$

then

$$
\dot V = -\dot q^T P \dot q < 0 \quad \forall\,\dot q \neq 0.
$$

Therefore,

$$
\dot q(t) \rightarrow 0,
$$

globally and asymptotically.

Important: nothing in this construction regulates $q$.  
The system comes to rest at some constant configuration determined by initial conditions.

This prototype removes motion — it does not choose the final position.

### 2.2.1.2 - Tracking problem

Now modify the objective.  
Instead of driving $\dot q \to 0$, suppose we want to track a reference velocity $\dot q_r(t)$.

Define the velocity tracking error

$$
\delta \dot q = \dot q - \dot q_r.
$$

A natural extension of the previous prototype is

$$
V(\delta \dot q) = \frac{1}{2}\delta \dot q^T M(q)\delta \dot q.
$$

This function is positive definite in the tracking error and zero only when
$$
\delta \dot q = 0.
$$

The difference from the regulation case is subtle but important:

- This is no longer the true kinetic energy of the plant.
- It is an energy-like measure of tracking error.

As a result, the clean work–energy simplification no longer holds automatically.  
Additional terms appear in $\dot V$ that must be handled through feedforward compensation and damping injection.

With appropriate control design (cancellation of nonlinear terms + dissipative feedback), one can again achieve

$$
\delta \dot q(t) \rightarrow 0
$$

But just as before, this guarantees convergence of velocities, not configuration.


**Key Insight**

Velocity-based Lyapunov functions are structurally powerful because:

- kinetic energy is naturally positive definite,
- its derivative often simplifies dramatically,
- dissipation directly produces stability.

However, they answer only one question:

Can motion be eliminated?

To control where the system ultimately settles, configuration errors must appear explicitly in the Lyapunov function. That is the motivation for the next prototype class.

## 2.2.2 - Rigid-Body Detumbling

Let us now apply the velocity-based Lyapunov prototype to a spacecraft.

Instead of generalized coordinates $q$, we work directly with the body-frame angular velocity

$$
\boldsymbol{\omega} \in \mathbb{R}^3.
$$

The detumbling objective is straightforward:

$$
\boldsymbol{\omega} \rightarrow \mathbf{0},
$$

with no requirement on the final attitude.

This situation occurs immediately after separation, during safe-mode recovery, or whenever rotation must be arrested before higher-level control is activated.

**<ins>Rigid-Body Rotational Dynamics</ins>**

The rotational equations of motion of a rigid spacecraft expressed in the body frame are

$$
[I]\dot{\boldsymbol{\omega}} = - [\tilde{\boldsymbol{\omega}}][I]\boldsymbol{\omega} + \boldsymbol{u},
$$

where:

- $[I]=[I]^T>0$ is the constant inertia matrix,
- $[\tilde{\boldsymbol{\omega}}]$ is the skew-symmetric cross-product matrix,
- $\boldsymbol{u}$ is the applied control torque.

The nonlinear term
$$
- [\tilde{\boldsymbol{\omega}}][I]\boldsymbol{\omega}
$$
is gyroscopic. It redistributes angular momentum but does not create or dissipate energy.

**<ins>Lyapunov Candidate: Rotational Kinetic Energy</ins>**

Following the velocity-based prototype, choose

$$
V(\boldsymbol{\omega}) = \frac{1}{2} \boldsymbol{\omega}^T [I]\boldsymbol{\omega}.
$$

This function is:

- positive definite in $\boldsymbol{\omega}$,
- zero only when $\boldsymbol{\omega}=\mathbf{0}$,
- radially unbounded in angular velocity.

It measures how much rotational motion the spacecraft has.

**<ins>Lyapunov Rate</ins>**

Differentiate along system trajectories:

$$
\dot V = \boldsymbol{\omega}^T [I]\dot{\boldsymbol{\omega}}.
$$

Substituting the dynamics gives

$$
\dot V = \boldsymbol{\omega}^T \left(-[\tilde{\boldsymbol{\omega}}][I]\boldsymbol{\omega} + \boldsymbol{u}\right).
$$

Because $[\tilde{\boldsymbol{\omega}}]$ is skew-symmetric,

$$
\boldsymbol{\omega}^T[\tilde{\boldsymbol{\omega}}] = \mathbf{0}^T,
$$

so the gyroscopic term vanishes identically. The energy rate reduces to

$$
\dot V = \boldsymbol{\omega}^T \boldsymbol{u}.
$$

Only applied torque can change rotational kinetic energy.

**<ins>Damping Injection</ins>**

Choose a simple dissipative torque:

$$
\boldsymbol{u} = -[P]\boldsymbol{\omega}, \qquad [P]=[P]^T>0.
$$

Then

$$
\dot V = - \boldsymbol{\omega}^T [P]\boldsymbol{\omega} < 0 \quad \forall\,\boldsymbol{\omega}\neq\mathbf{0}.
$$

**<ins>Stability Conclusion</ins>**

The Lyapunov function is positive definite and its derivative is negative definite.

Therefore:

- $\boldsymbol{\omega}(t)\rightarrow \mathbf{0}$ asymptotically,
- the zero-rate equilibrium is globally asymptotically stable in angular velocity,
- the result holds for any initial angular velocity.

The final attitude is unconstrained.

**<ins>Invariant Set Interpretation</ins>**

The closed-loop system converges to

$$
\Omega =
\{
(\text{attitude},\boldsymbol{\omega}) 
:
\boldsymbol{\omega}=\mathbf{0}
\}.
$$

Detumbling guarantees that rotation stops. It does not guarantee pointing accuracy.

**<ins>Structural Insight</ins>**

This controller works because:

- kinetic energy is a natural positive-definite measure of motion,
- gyroscopic terms do no work,
- damping directly removes energy.

No cancellation of nonlinear dynamics is required.  
No attitude parameterization is needed.  
The result is global in angular velocity.

Rigid-body detumbling is therefore the cleanest spacecraft example of the velocity-based Lyapunov prototype.

## 2.2.3 - Limitations of Rate-only Lyapunov Functions

Velocity-based Lyapunov functions are powerful, but they solve only part of the control problem.

They guarantee

$$
\dot q(t) \rightarrow 0
\quad \text{or} \quad
\boldsymbol{\omega}(t) \rightarrow \mathbf{0},
$$

which means motion eventually stops.

But stopping motion is not the same as reaching the correct configuration.

If a spacecraft begins tumbling and we apply detumbling control, it will eventually rotate at zero angular velocity. However, the final attitude depends entirely on the initial condition. The controller has removed kinetic energy, but it has not shaped where the system settles.

Mathematically, rate-only Lyapunov functions are positive definite in velocity but independent of configuration. Once the velocity converges to zero, the Lyapunov function cannot distinguish between different constant configurations.

The invariant set for rate-based control is

$$
\Omega = \{ (q,\dot q) : \dot q = 0 \}.
$$

Every constant configuration belongs to this set. The Lyapunov function does not prefer one equilibrium over another.

This reveals the structural limitation:

- Velocity-based Lyapunov functions regulate motion.
- They do not regulate position or attitude.
- They cannot enforce a unique equilibrium in configuration space.

To uniquely determine where the system settles, the Lyapunov function must penalize configuration error explicitly.

In mechanical systems, this corresponds to adding a potential-energy-like term.  
In spacecraft attitude control, this means introducing an attitude error measure into the Lyapunov function.

Rate regulation removes motion.  
Configuration terms determine where the motion stops.

This is the motivation for state-based Lyapunov functions.

## 2.2.4 - State-Based Lyapunov Functions

Velocity-based Lyapunov functions remove motion.  
They ensure $\dot q \rightarrow 0$, but they do not determine where the system settles.

To regulate configuration itself, the Lyapunov function must measure configuration error directly.

Slide 21 introduces this idea through a simple physical analogy: a linear spring.

The energy stored in a spring with stiffness $k$ and displacement $x$ is

$$
V(x) = \frac{1}{2} k x^2.
$$

This function has the exact properties required of a Lyapunov candidate:

- It is positive definite in the displacement.
- It is zero only at the equilibrium.
- It grows as the error grows.

In control terms, this energy behaves like a “restoring potential” centered at the desired configuration.

If we define a configuration error

$$
e_q = q - q_r,
$$

then a natural state-based Lyapunov candidate is

$$
V(e_q) = \frac{1}{2} e_q^T K e_q,
\qquad K = K^T > 0.
$$

This is simply the multidimensional generalization of the spring energy shown in Slide 21.

The key conceptual shift is this:

- Velocity-based functions measure motion.
- State-based functions measure displacement from the target.

In many robotic and mechanical systems, the state rate $\dot q$ can be treated as a control variable through a lower-level servo loop. In that case, configuration regulation can be achieved by shaping this potential-like energy directly.

However, a purely state-based Lyapunov function does not automatically regulate velocity. It enforces where the system should settle, but additional structure is required to ensure motion decays.

For full stabilization of mechanical systems, kinetic-energy-like and potential-energy-like terms are typically combined:

$$
V(q,\dot q) = \frac{1}{2}\dot q^T M(q)\dot q + \Phi(e_q),
$$

where $\Phi(e_q)$ plays the role of a generalized spring energy.

State-based Lyapunov functions therefore provide the missing ingredient left by rate-only methods: they define the unique configuration toward which the system should converge.

# 2.3 - Geometry of Attitude Representation in Lyapunov Design

State-based Lyapunov functions for attitude control need a **potential-like** term that measures “how far” the attitude is from the desired attitude.

The catch is that “attitude error” is not a single unique vector quantity. It depends on the attitude representation you choose (Euler angles, CRP, MRP, quaternions). Different representations lead to different potential functions, and those choices affect:

- whether the Lyapunov function is **globally well-defined**,
- whether it is **continuous everywhere**,
- and how clean the resulting feedback law looks.

The Week 2 slides (Slides 22–24) give several commonly used potential functions and show what their Lyapunov rates look like under the kinematics. (Week 2 slides: 2 - Overview of Lyapunov Stability Theory, Slides 22–24)

## 2.3.1 - Euler Angle Potential Function

Let the attitude error be represented by Euler angle error vector
$$
\boldsymbol{\theta} = (\theta_1,\theta_2,\theta_3)^T.
$$

A straightforward “spring energy” candidate is
$$
V(\boldsymbol{\theta}) = \frac{1}{2}\boldsymbol{\theta}^T[K]\boldsymbol{\theta},
\qquad [K]=[K]^T>0.
$$

Using Euler angle kinematics of the form
$$
\dot{\boldsymbol{\theta}} = [B(\boldsymbol{\theta})]\boldsymbol{\omega},
$$
the Lyapunov rate becomes
$$
\dot V = \boldsymbol{\omega}^T [B(\boldsymbol{\theta})]^T [K]\boldsymbol{\theta}.
$$

For tracking, the same potential is used but with the tracking rate error
$$
\delta\boldsymbol{\omega}=\boldsymbol{\omega}-\boldsymbol{\omega}_r,
\qquad
\dot{\boldsymbol{\theta}}=[B(\boldsymbol{\theta})]\delta\boldsymbol{\omega}.
$$

Key point from the slide:
There is no algebraic distinction between regulator vs tracking at the level of the **position-based potential**. The difference shows up later when you define the rate feedback term. (Slide 22)

Practical note:
Euler angles can suffer from singularities. So even if the quadratic potential is nice locally, it is not necessarily a good global choice.


## 2.3.2 - CRP Potential Function

Let the attitude error be represented using the Classical Rodrigues Parameters (Gibbs vector)
$$
\boldsymbol{q}\in\mathbb{R}^3,
\qquad q^2 = \boldsymbol{q}^T\boldsymbol{q}.
$$

A “brute force” spring-like candidate is
$$
V(\boldsymbol{q}) = \boldsymbol{q}^T[K]\boldsymbol{q}.
$$

Differentiating and substituting CRP kinematics produces a more complicated expression (the slide shows it reduces into a nonlinear structure), which tends to drive you toward nonlinear feedback laws. (Slide 23)

The slide then presents a more elegant choice:
$$
V(\boldsymbol{q}) = K \ln\left(1+\boldsymbol{q}^T\boldsymbol{q}\right),
$$
whose derivative simplifies to
$$
\dot V = \boldsymbol{\omega}^T (K\boldsymbol{q}).
$$

This is the main payoff:
A logarithmic potential can produce a surprisingly clean Lyapunov rate, which leads naturally to a simple linear-looking attitude feedback choice such as
$$
\boldsymbol{\omega} = -[K]\boldsymbol{q},
$$
in a servo-law development context. (Slide 23)

Practical note:
CRPs become singular at 180 degrees. So the potential can be useful, but global behavior must be treated carefully.

## 2.3.3 - MRP Potential Function

Let the attitude error be represented using MRPs
$$
\boldsymbol{\sigma}\in\mathbb{R}^3.
$$

The slide proposes the potential
$$
V(\boldsymbol{\sigma}) = 2K \ln\left(1+\boldsymbol{\sigma}^T\boldsymbol{\sigma}\right).
$$

Its Lyapunov rate simplifies to
$$
\dot V = \boldsymbol{\omega}^T (K\boldsymbol{\sigma}).
$$

Important detail from the slide:
If you switch to the **shadow MRP set** on the $\sigma^2 = 1$ surface, then this Lyapunov function can be treated as continuous, enabling globally stabilizing feedback laws by switching between the original and shadow set. (Slide 24)

Interpretation:
MRPs give you a clean 3-parameter representation, and the shadow set trick is how you avoid the “long way around” singular behavior.

## 2.3.4 - Euler Parameters / Quaternion Potential Function

Let the attitude be represented with Euler parameters (unit quaternion)
$$
\boldsymbol{\beta} = (\beta_0,\beta_1,\beta_2,\beta_3)^T,
\qquad \|\boldsymbol{\beta}\|=1.
$$

The slide considers regulation to the “ideal attitude”
$$
\hat{\boldsymbol{\beta}}=
\begin{bmatrix}
1\\0\\0\\0
\end{bmatrix}.
$$

A simple quadratic potential is
$$
V(\boldsymbol{\beta}) = K(\boldsymbol{\beta}-\hat{\boldsymbol{\beta}})^T(\boldsymbol{\beta}-\hat{\boldsymbol{\beta}}).
$$

Using quaternion kinematics
$$
\dot{\boldsymbol{\beta}} = [B(\boldsymbol{\beta})]\boldsymbol{\omega},
$$
the slide shows the Lyapunov rate becomes
$$
\dot V = K\boldsymbol{\omega}^T [B(\boldsymbol{\beta})]^T(\boldsymbol{\beta}-\hat{\boldsymbol{\beta}}).
$$

A key identity used is
$$
[B(\boldsymbol{\beta})]^T\boldsymbol{\beta}=0,
$$
which yields a simplified form
$$
\dot V = \boldsymbol{\omega}^T (K\boldsymbol{\epsilon}),
\qquad
\boldsymbol{\epsilon}=
\begin{bmatrix}
\beta_1\\\beta_2\\\beta_3
\end{bmatrix}.
$$

Practical note from the slide:
This stabilizes to $\beta_0 = \pm 1$, which corresponds to the same physical attitude (quaternions double-cover $SO(3)$). However, this alone does not guarantee whether the “short” or “long” rotation is taken. (Slide 24)

## 2.3.5 - Concluding Remarks

Across all representations, the pattern is consistent:

1. Choose a potential-like $V$ that is positive definite in the chosen attitude error coordinates.
2. Differentiate using the kinematics $\dot{\text{(attitude error)}} = [B(\cdot)](\text{rate})$.
3. Look for a form of $\dot V$ that can be made negative by a clean feedback choice.

The representation matters because it determines:
- whether your “error coordinates” are globally valid,
- whether $V$ stays continuous everywhere,
- and how simple (or messy) $\dot V$ becomes after substitution.

This is why “prototype Lyapunov functions” are tightly coupled to attitude representation choice in spacecraft control.

---

Notes compiled and structured by John Gracious